<a href="https://colab.research.google.com/github/PETEROA/ML-Optimization-Daily/blob/main/day_24_distributed_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Distributed Training

Here i attempt to cover distributed training, scaling training across multiple GPUs and machines. Single GPU training hits fundamental limits: memory caps model size, compute caps training speed. Distributed training breaks these limits through parallelism strategies. I cover data parallelism (same model, different data), model parallelism (different model parts, same data), DistributedDataParallel (DDP) for multi-GPU, Fully Sharded Data Parallel (FSDP) for memory efficiency, and communication primitives (all-reduce, all-gather). I implement these patterns from scratch, understand their tradeoffs, and learn When to use each. This is essential for training large models.

In [75]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DataParallel, DistributedDataParallel
from torch.utils.data import DataLoader, Dataset, DistributedSampler
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, List, Dict, Tuple, Callable, Any
from dataclasses import dataclass
from abc import ABC, abstractmethod
import os
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")

Using device: cuda
GPU: Tesla T4
GPU Count: 1


In [76]:
# Memory calculator for distributed training

@dataclass
class DistributedMemoryEstimate:
    """Estimate memory requirements for distributed training."""
    params_gb: float
    gradients_gb: float
    optimizer_gb: float
    activations_gb: float
    total_gb: float


def estimate_training_memory(num_params: int, batch_size: int, seq_len: int,
                              hidden_dim: int, num_layers: int,
                              precision: str = 'fp32') -> DistributedMemoryEstimate:
    """
    Estimate memory requirements for training.

    Args:
        num_params: Number of model parameters
        batch_size: Training batch size
        seq_len: Sequence length (for transformers)
        hidden_dim: Hidden dimension
        num_layers: Number of layers
        precision: 'fp32' or 'fp16'
    """
    bytes_per_param = 4 if precision == 'fp32' else 2

    # Parameters
    params_bytes = num_params * bytes_per_param

    # Gradients (always FP32 for accumulation)
    gradients_bytes = num_params * 4

    # Optimizer states (Adam: momentum + variance)
    optimizer_bytes = num_params * 4 * 2

    # Activations (rough estimate for transformers)
    # Each layer stores: batch × seq × hidden for attention + FFN
    activation_per_layer = batch_size * seq_len * hidden_dim * bytes_per_param * 4
    activations_bytes = activation_per_layer * num_layers

    total_bytes = params_bytes + gradients_bytes + optimizer_bytes + activations_bytes

    return DistributedMemoryEstimate(
        params_gb=params_bytes / 1e9,
        gradients_gb=gradients_bytes / 1e9,
        optimizer_gb=optimizer_bytes / 1e9,
        activations_gb=activations_bytes / 1e9,
        total_gb=total_bytes / 1e9
    )


# Example: GPT-style models
print("Memory Requirements for Various Model Sizes")
print("=" * 70)

models = [
    ("GPT-2 Small", 125e6, 768, 12),
    ("GPT-2 Medium", 355e6, 1024, 24),
    ("GPT-2 Large", 774e6, 1280, 36),
    ("GPT-3 Small", 1.3e9, 2048, 24),
    ("GPT-3 Medium", 6.7e9, 4096, 32),
    ("GPT-3 Large", 175e9, 12288, 96),
]

print(f"\n{'Model':<15} {'Params':>12} {'Total (GB)':>12} {'Params':>10} {'Grad':>10} {'Optim':>10} {'Act':>10}")
print("-" * 85)

for name, params, hidden, layers in models:
    mem = estimate_training_memory(int(params), batch_size=1, seq_len=2048,
                                   hidden_dim=hidden, num_layers=layers, precision='fp16')
    print(f"{name:<15} {params/1e9:>11.1f}B {mem.total_gb:>11.1f} "
          f"{mem.params_gb:>10.1f} {mem.gradients_gb:>10.1f} "
          f"{mem.optimizer_gb:>10.1f} {mem.activations_gb:>10.1f}")

Memory Requirements for Various Model Sizes

Model                 Params   Total (GB)     Params       Grad      Optim        Act
-------------------------------------------------------------------------------------
GPT-2 Small             0.1B         1.9        0.2        0.5        1.0        0.2
GPT-2 Medium            0.4B         5.4        0.7        1.4        2.8        0.4
GPT-2 Large             0.8B        11.6        1.5        3.1        6.2        0.8
GPT-3 Small             1.3B        19.0        2.6        5.2       10.4        0.8
GPT-3 Medium            6.7B        95.9       13.4       26.8       53.6        2.1
GPT-3 Large           175.0B      2469.3      350.0      700.0     1400.0       19.3


Communication Primitives

Distributed training requires efficient communication between GPUs.

In [77]:
# Simulate communication primitives
# (In real distributed training, these use NCCL/Gloo backends)

class SimulatedDistributed:
    """
    Simulate distributed communication primitives.

    In real training, these would use:
    - NCCL for GPU-to-GPU
    - Gloo for CPU
    - MPI for HPC clusters
    """

    def __init__(self, world_size: int):
        self.world_size = world_size

    def all_reduce(self, tensors: List[torch.Tensor], op: str = 'sum') -> torch.Tensor:
        """
        All-reduce: Combine tensors from all ranks, result on all ranks.

        Before: [T0, T1, T2, T3]  (one per GPU)
        After:  [T0+T1+T2+T3, T0+T1+T2+T3, T0+T1+T2+T3, T0+T1+T2+T3]

        Used for: Gradient synchronization in data parallelism
        """
        if op == 'sum':
            result = sum(tensors)
        elif op == 'mean':
            result = sum(tensors) / len(tensors)
        else:
            raise ValueError(f"Unknown op: {op}")

        return [result.clone() for _ in range(self.world_size)]

    def all_gather(self, tensors: List[torch.Tensor]) -> List[List[torch.Tensor]]:
        """
        All-gather: Gather tensors from all ranks to all ranks.

        Before: [T0, T1, T2, T3]
        After:  [[T0,T1,T2,T3], [T0,T1,T2,T3], [T0,T1,T2,T3], [T0,T1,T2,T3]]

        Used for: Gathering model weights in FSDP
        """
        gathered = [t.clone() for t in tensors]
        return [gathered.copy() for _ in range(self.world_size)]

    def reduce_scatter(self, tensors: List[torch.Tensor]) -> List[torch.Tensor]:
        """
        Reduce-scatter: Reduce and scatter result (each rank gets 1/N).

        Before: Each rank has full gradient
        After:  Each rank has 1/N of reduced gradient

        Used for: Efficient gradient sync in FSDP
        """
        # Sum all tensors
        total = sum(tensors)

        # Split into chunks
        chunk_size = total.numel() // self.world_size
        flat = total.flatten()

        chunks = []
        for i in range(self.world_size):
            start = i * chunk_size
            end = start + chunk_size if i < self.world_size - 1 else flat.numel()
            chunks.append(flat[start:end].clone())

        return chunks

    def broadcast(self, tensor: torch.Tensor, src: int) -> List[torch.Tensor]:
        """
        Broadcast: Send tensor from source to all ranks.

        Used for: Parameter initialization, syncing buffers
        """
        return [tensor.clone() for _ in range(self.world_size)]

In [78]:
# Demonstrate communication primitives

print("Communication Primitives Demonstration")
print("=" * 60)

world_size = 4
sim_dist = SimulatedDistributed(world_size)

# Create different tensors for each "GPU"
tensors = [torch.tensor([i + 1.0, i + 2.0, i + 3.0]) for i in range(world_size)]

print(f"\nInitial tensors (one per GPU):")
for i, t in enumerate(tensors):
    print(f"  GPU {i}: {t.tolist()}")

# All-reduce (sum)
print(f"\nAfter all_reduce (sum):")
reduced = sim_dist.all_reduce(tensors, op='sum')
for i, t in enumerate(reduced):
    print(f"  GPU {i}: {t.tolist()}")

# All-gather
print(f"\nAfter all_gather:")
gathered = sim_dist.all_gather(tensors)
for i, g in enumerate(gathered):
    print(f"  GPU {i}: {[t.tolist() for t in g]}")

# Broadcast
print(f"\nAfter broadcast from GPU 0:")
broadcasted = sim_dist.broadcast(tensors[0], src=0)
for i, t in enumerate(broadcasted):
    print(f"  GPU {i}: {t.tolist()}")

Communication Primitives Demonstration

Initial tensors (one per GPU):
  GPU 0: [1.0, 2.0, 3.0]
  GPU 1: [2.0, 3.0, 4.0]
  GPU 2: [3.0, 4.0, 5.0]
  GPU 3: [4.0, 5.0, 6.0]

After all_reduce (sum):
  GPU 0: [10.0, 14.0, 18.0]
  GPU 1: [10.0, 14.0, 18.0]
  GPU 2: [10.0, 14.0, 18.0]
  GPU 3: [10.0, 14.0, 18.0]

After all_gather:
  GPU 0: [[1.0, 2.0, 3.0], [2.0, 3.0, 4.0], [3.0, 4.0, 5.0], [4.0, 5.0, 6.0]]
  GPU 1: [[1.0, 2.0, 3.0], [2.0, 3.0, 4.0], [3.0, 4.0, 5.0], [4.0, 5.0, 6.0]]
  GPU 2: [[1.0, 2.0, 3.0], [2.0, 3.0, 4.0], [3.0, 4.0, 5.0], [4.0, 5.0, 6.0]]
  GPU 3: [[1.0, 2.0, 3.0], [2.0, 3.0, 4.0], [3.0, 4.0, 5.0], [4.0, 5.0, 6.0]]

After broadcast from GPU 0:
  GPU 0: [1.0, 2.0, 3.0]
  GPU 1: [1.0, 2.0, 3.0]
  GPU 2: [1.0, 2.0, 3.0]
  GPU 3: [1.0, 2.0, 3.0]


In [79]:
# Communication cost analysis

def analyze_communication_cost(num_params: int, world_size: int,
                                bandwidth_gbps: float = 200) -> Dict:
    """
    Analyze communication costs for different strategies.

    Args:
        num_params: Number of model parameters
        world_size: Number of GPUs
        bandwidth_gbps: Network bandwidth (GB/s)
    """
    param_bytes = num_params * 4  # FP32
    param_gb = param_bytes / 1e9

    # All-reduce: Ring algorithm sends 2 * (N-1)/N * data
    all_reduce_data = 2 * (world_size - 1) / world_size * param_gb
    all_reduce_time = all_reduce_data / bandwidth_gbps

    # All-gather: Each GPU sends its shard to all others
    all_gather_data = param_gb * (world_size - 1) / world_size
    all_gather_time = all_gather_data / bandwidth_gbps

    # Reduce-scatter: Like all-reduce but result is scattered
    reduce_scatter_data = (world_size - 1) / world_size * param_gb
    reduce_scatter_time = reduce_scatter_data / bandwidth_gbps

    return {
        'param_gb': param_gb,
        'all_reduce_gb': all_reduce_data,
        'all_reduce_ms': all_reduce_time * 1000,
        'all_gather_gb': all_gather_data,
        'all_gather_ms': all_gather_time * 1000,
        'reduce_scatter_gb': reduce_scatter_data,
        'reduce_scatter_ms': reduce_scatter_time * 1000
    }


print("Communication Cost Analysis")
print("=" * 70)
print("(Assuming 200 GB/s NVLink bandwidth)\n")

for params, name in [(100e6, "100M"), (1e9, "1B"), (10e9, "10B")]:
    print(f"Model: {name} parameters")
    for ws in [2, 4, 8]:
        costs = analyze_communication_cost(int(params), ws)
        print(f"  {ws} GPUs: all_reduce={costs['all_reduce_ms']:.1f}ms, "
              f"all_gather={costs['all_gather_ms']:.1f}ms")
    print()

Communication Cost Analysis
(Assuming 200 GB/s NVLink bandwidth)

Model: 100M parameters
  2 GPUs: all_reduce=2.0ms, all_gather=1.0ms
  4 GPUs: all_reduce=3.0ms, all_gather=1.5ms
  8 GPUs: all_reduce=3.5ms, all_gather=1.8ms

Model: 1B parameters
  2 GPUs: all_reduce=20.0ms, all_gather=10.0ms
  4 GPUs: all_reduce=30.0ms, all_gather=15.0ms
  8 GPUs: all_reduce=35.0ms, all_gather=17.5ms

Model: 10B parameters
  2 GPUs: all_reduce=200.0ms, all_gather=100.0ms
  4 GPUs: all_reduce=300.0ms, all_gather=150.0ms
  8 GPUs: all_reduce=350.0ms, all_gather=175.0ms



Data Parallelism

The simplest form: replicate model on each GPU, split data.

In [57]:
class SimulatedDataParallel:
    """
    Simulated Data Parallel training.

    Each GPU has:
    - Full copy of model
    - 1/N of each batch

    Synchronization:
    - Forward: Independent
    - Backward: Independent
    - Update: All-reduce gradients, then update
    """

    def __init__(self, model: nn.Module, world_size: int):
        self.world_size = world_size

        # Replicate model to each "GPU"
        self.models = []
        for _ in range(world_size):
            replica = type(model)()  # Create new instance
            replica.load_state_dict(model.state_dict())
            self.models.append(replica)

        self.dist = SimulatedDistributed(world_size)

    def forward(self, batch: torch.Tensor) -> List[torch.Tensor]:
        """
        Forward pass on all replicas.
        """
        # Split batch across GPUs
        batch_size = batch.size(0)
        chunk_size = batch_size // self.world_size

        outputs = []
        for i, model in enumerate(self.models):
            start = i * chunk_size
            end = start + chunk_size if i < self.world_size - 1 else batch_size
            chunk = batch[start:end]

            output = model(chunk)
            outputs.append(output)

        return outputs

    def backward_and_sync(self, losses: List[torch.Tensor]):
        """
        Backward pass and gradient synchronization.
        """
        # Backward on each replica
        for loss in losses:
            loss.backward()

        # All-reduce gradients
        for name, param in self.models[0].named_parameters():
            grads = [m.get_parameter(name).grad for m in self.models]

            # Average gradients (all-reduce with mean)
            avg_grads = self.dist.all_reduce(grads, op='mean')

            # Update all replicas with averaged gradient
            for i, model in enumerate(self.models):
                model.get_parameter(name).grad = avg_grads[i]

    def get_model(self) -> nn.Module:
        """Return one model (they should all be synchronized)."""
        return self.models[0]

In [58]:
# Demonstrate data parallelism

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 5)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


print("Simulated Data Parallelism")
print("=" * 50)

model = SimpleModel()
world_size = 4
batch_size = 16  # Will be split into 4 chunks of 4

dp = SimulatedDataParallel(model, world_size)

# Create batch
batch = torch.randn(batch_size, 10)
targets = torch.randint(0, 5, (batch_size,))

# Forward
outputs = dp.forward(batch)
print(f"Input batch size: {batch_size}")
print(f"Outputs per GPU: {[o.shape[0] for o in outputs]}")

# Compute losses
criterion = nn.CrossEntropyLoss()
chunk_size = batch_size // world_size
losses = []
for i, output in enumerate(outputs):
    start = i * chunk_size
    end = start + chunk_size if i < world_size - 1 else batch_size
    loss = criterion(output, targets[start:end])
    losses.append(loss)
    print(f"GPU {i} loss: {loss.item():.4f}")

# Backward and sync
dp.backward_and_sync(losses)

# Verify gradients are synchronized
print("\nGradients synchronized across GPUs:")
for name, _ in dp.models[0].named_parameters():
    grads = [m.get_parameter(name).grad for m in dp.models]
    all_equal = all(torch.allclose(grads[0], g) for g in grads[1:])
    print(f"  {name}: {'✓' if all_equal else '✗'}")

Simulated Data Parallelism
Input batch size: 16
Outputs per GPU: [4, 4, 4, 4]
GPU 0 loss: 1.6002
GPU 1 loss: 1.4574
GPU 2 loss: 1.6334
GPU 3 loss: 1.6647

Gradients synchronized across GPUs:
  fc1.weight: ✓
  fc1.bias: ✓
  fc2.weight: ✓
  fc2.bias: ✓


DistributedDataParallel (DDP)

PyTorch's optimized data parallelism with overlapped communication.

In [59]:
class SimulatedDDP:
    """
    Simulated DistributedDataParallel.

    Key optimizations over naive DP:
    1. Gradient bucketing: Group small gradients for efficient all-reduce
    2. Overlap: Start all-reduce while backward is still running
    3. No parameter broadcast: Each GPU maintains its own copy
    """

    def __init__(self, model: nn.Module, world_size: int,
                 bucket_size_mb: float = 25):
        self.model = model
        self.world_size = world_size
        self.bucket_size_bytes = int(bucket_size_mb * 1e6)

        # Create gradient buckets
        self.buckets = self._create_buckets()

        # Register backward hooks for overlapped communication
        self._register_hooks()

        self.dist = SimulatedDistributed(world_size)

    def _create_buckets(self) -> List[List[str]]:
        """
        Group parameters into buckets for efficient all-reduce.

        Parameters are bucketed in reverse order (last layers first)
        because backward pass computes gradients in reverse order.
        """
        buckets = []
        current_bucket = []
        current_size = 0

        # Reverse order for backward pass
        params = list(self.model.named_parameters())
        params.reverse()

        for name, param in params:
            param_size = param.numel() * param.element_size()

            if current_size + param_size > self.bucket_size_bytes and current_bucket:
                buckets.append(current_bucket)
                current_bucket = []
                current_size = 0

            current_bucket.append(name)
            current_size += param_size

        if current_bucket:
            buckets.append(current_bucket)

        return buckets

    def _register_hooks(self):
        """
        Register hooks to trigger all-reduce when bucket is ready.
        """
        # In real DDP, hooks trigger async all-reduce
        # Here we just track which gradients are ready
        self.ready_params = set()

    def all_reduce_gradients(self):
        """
        All-reduce gradients bucket by bucket.

        In real DDP, this happens asynchronously as buckets fill up.
        """
        for bucket in self.buckets:
            # Flatten gradients in bucket
            grads = []
            for name in bucket:
                param = dict(self.model.named_parameters())[name]
                if param.grad is not None:
                    grads.append(param.grad.flatten())

            if grads:
                flat_grad = torch.cat(grads)

                # Simulate all-reduce across world_size replicas
                # In real DDP, this is async and overlapped with backward
                reduced = flat_grad / self.world_size  # Average

                # Unflatten and update
                offset = 0
                for name in bucket:
                    param = dict(self.model.named_parameters())[name]
                    if param.grad is not None:
                        numel = param.grad.numel()
                        param.grad.data = reduced[offset:offset + numel].view_as(param.grad)
                        offset += numel


# Explain DDP bucket strategy
print("DDP Bucketing Strategy")
print("=" * 60)

class LargerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = F.relu(F.max_pool2d(self.conv3(x), 2))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = LargerModel()
ddp = SimulatedDDP(model, world_size=4, bucket_size_mb=0.1)  # Small buckets for demo

print(f"\nModel parameters:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.numel():,} params ({param.numel() * 4 / 1e6:.3f} MB)")

print(f"\nGradient buckets ({len(ddp.buckets)} buckets):")
for i, bucket in enumerate(ddp.buckets):
    total_params = sum(dict(model.named_parameters())[n].numel() for n in bucket)
    print(f"  Bucket {i}: {bucket} ({total_params:,} params)")

DDP Bucketing Strategy

Model parameters:
  conv1.weight: 1,728 params (0.007 MB)
  conv1.bias: 64 params (0.000 MB)
  conv2.weight: 73,728 params (0.295 MB)
  conv2.bias: 128 params (0.001 MB)
  conv3.weight: 294,912 params (1.180 MB)
  conv3.bias: 256 params (0.001 MB)
  fc1.weight: 2,097,152 params (8.389 MB)
  fc1.bias: 512 params (0.002 MB)
  fc2.weight: 5,120 params (0.020 MB)
  fc2.bias: 10 params (0.000 MB)

Gradient buckets (7 buckets):
  Bucket 0: ['fc2.bias', 'fc2.weight', 'fc1.bias'] (5,642 params)
  Bucket 1: ['fc1.weight'] (2,097,152 params)
  Bucket 2: ['conv3.bias'] (256 params)
  Bucket 3: ['conv3.weight'] (294,912 params)
  Bucket 4: ['conv2.bias'] (128 params)
  Bucket 5: ['conv2.weight'] (73,728 params)
  Bucket 6: ['conv1.bias', 'conv1.weight'] (1,792 params)


In [60]:
# DDP training loop pattern

def ddp_training_pattern():
    """
    Show typical DDP training pattern (pseudo-code for multi-GPU).
    """
    pattern = '''
# DDP Training Pattern
# ====================

import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def setup(rank, world_size):
    """Initialize distributed process group."""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

def cleanup():
    dist.destroy_process_group()

def train(rank, world_size):
    setup(rank, world_size)

    # Create model and move to GPU
    model = MyModel().to(rank)

    # Wrap with DDP
    ddp_model = DDP(model, device_ids=[rank])

    # Use DistributedSampler for data
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, sampler=sampler, batch_size=batch_size)

    optimizer = torch.optim.Adam(ddp_model.parameters())

    for epoch in range(num_epochs):
        sampler.set_epoch(epoch)  # Important for shuffling

        for batch in dataloader:
            optimizer.zero_grad()

            # Forward (each GPU processes 1/N of batch)
            output = ddp_model(batch)
            loss = criterion(output, target)

            # Backward (gradients auto-synced by DDP)
            loss.backward()

            # Update (all GPUs have same gradients)
            optimizer.step()

    cleanup()

# Launch with torch.multiprocessing
if __name__ == "__main__":
    world_size = torch.cuda.device_count()
    torch.multiprocessing.spawn(train, args=(world_size,), nprocs=world_size)
'''
    return pattern

print(ddp_training_pattern())


# DDP Training Pattern
# ====================

import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def setup(rank, world_size):
    """Initialize distributed process group."""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

def cleanup():
    dist.destroy_process_group()

def train(rank, world_size):
    setup(rank, world_size)
    
    # Create model and move to GPU
    model = MyModel().to(rank)
    
    # Wrap with DDP
    ddp_model = DDP(model, device_ids=[rank])
    
    # Use DistributedSampler for data
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, sampler=sampler, batch_size=batch_size)
    
    optimizer = torch.optim.Adam(ddp_model.parameters())
    
    for epoch in range(num_epochs):
        sampler.

Model Parallelism

Split model across GPUs when model doesn't fit on one GPU.

In [61]:
class TensorParallelLinear(nn.Module):
    """
    Tensor Parallel Linear Layer.

    Splits weight matrix across GPUs:
    - Column parallel: Split output features
    - Row parallel: Split input features

    For MLP: Column parallel \u2192 Row parallel (no communication between)
    """

    def __init__(self, in_features: int, out_features: int,
                 world_size: int, parallel_mode: str = 'column'):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.world_size = world_size
        self.parallel_mode = parallel_mode

        if parallel_mode == 'column':
            # Each GPU has full input, partial output
            self.local_out = out_features // world_size
            self.weights = nn.ParameterList([
                nn.Parameter(torch.randn(self.local_out, in_features) * 0.01)
                for _ in range(world_size)
            ])
        else:  # row
            # Each GPU has partial input, full output
            self.local_in = in_features // world_size
            self.weights = nn.ParameterList([
                nn.Parameter(torch.randn(out_features, self.local_in) * 0.01)
                for _ in range(world_size)
            ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass with simulated tensor parallelism.
        """
        if self.parallel_mode == 'column':
            # Each GPU computes partial output
            outputs = [F.linear(x, w) for w in self.weights]
            # Concatenate results
            return torch.cat(outputs, dim=-1)
        else:
            # Split input
            chunks = torch.chunk(x, self.world_size, dim=-1)
            # Each GPU computes with its chunk
            outputs = [F.linear(chunk, w) for chunk, w in zip(chunks, self.weights)]
            # All-reduce (sum)
            return sum(outputs)


class LayerParallelModel(nn.Module):
    """
    Model with layers distributed across GPUs.

    Layer parallel (naive model parallel):
    - GPU 0: Layers 0-3
    - GPU 1: Layers 4-7
    - etc.

    Problem: Only one GPU active at a time!
    Solution: Pipeline parallelism (next section)
    """

    def __init__(self, layer_dims: List[int], num_gpus: int):
        super().__init__()

        self.num_gpus = num_gpus
        self.num_layers = len(layer_dims) - 1

        # Assign layers to GPUs
        layers_per_gpu = self.num_layers // num_gpus

        self.gpu_layers = nn.ModuleList()
        self.layer_to_gpu = {}

        for i in range(self.num_layers):
            gpu_id = min(i // layers_per_gpu, num_gpus - 1)
            layer = nn.Linear(layer_dims[i], layer_dims[i + 1])
            self.gpu_layers.append(layer)
            self.layer_to_gpu[i] = gpu_id

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        current_gpu = 0

        for i, layer in enumerate(self.gpu_layers):
            target_gpu = self.layer_to_gpu[i]

            # Simulate transfer if changing GPUs
            if target_gpu != current_gpu:
                # x = x.to(f'cuda:{target_gpu}')  # In real code
                current_gpu = target_gpu

            x = layer(x)

        return x

    def get_gpu_assignment(self) -> Dict[int, List[int]]:
        """Get which layers are on which GPU."""
        assignment = {i: [] for i in range(self.num_gpus)}
        for layer_id, gpu_id in self.layer_to_gpu.items():
            assignment[gpu_id].append(layer_id)
        return assignment

In [62]:
# Demonstrate model parallelism

print("Model Parallelism Demonstration")
print("=" * 60)

# Layer parallel
print("\n1. Layer Parallelism:")
layer_dims = [1024, 2048, 2048, 2048, 2048, 1024, 512, 10]
model = LayerParallelModel(layer_dims, num_gpus=4)

assignment = model.get_gpu_assignment()
for gpu_id, layers in assignment.items():
    print(f"  GPU {gpu_id}: Layers {layers}")

# Tensor parallel
print("\n2. Tensor Parallelism (Column):")
tp_linear = TensorParallelLinear(1024, 4096, world_size=4, parallel_mode='column')
x = torch.randn(8, 1024)
y = tp_linear(x)
print(f"  Input: {x.shape}")
print(f"  Output: {y.shape}")
print(f"  Each GPU weight shape: {tp_linear.weights[0].shape}")

print("\n3. Tensor Parallelism (Row):")
tp_linear_row = TensorParallelLinear(4096, 1024, world_size=4, parallel_mode='row')
x = torch.randn(8, 4096)
y = tp_linear_row(x)
print(f"  Input: {x.shape}")
print(f"  Output: {y.shape}")
print(f"  Each GPU weight shape: {tp_linear_row.weights[0].shape}")

Model Parallelism Demonstration

1. Layer Parallelism:
  GPU 0: Layers [0]
  GPU 1: Layers [1]
  GPU 2: Layers [2]
  GPU 3: Layers [3, 4, 5, 6]

2. Tensor Parallelism (Column):
  Input: torch.Size([8, 1024])
  Output: torch.Size([8, 4096])
  Each GPU weight shape: torch.Size([1024, 1024])

3. Tensor Parallelism (Row):
  Input: torch.Size([8, 4096])
  Output: torch.Size([8, 1024])
  Each GPU weight shape: torch.Size([1024, 1024])


Fully Sharded Data Parallel (FSDP)

Combines data parallelism with parameter sharding for memory efficiency.

In [63]:
class SimulatedFSDP:
    """
    Simulated Fully Sharded Data Parallel.

    Key idea: Shard parameters, gradients, AND optimizer states.

    Memory per GPU:
    - DDP: Full model + gradients + optimizer = 16P bytes
    - FSDP: 1/N of (model + gradients + optimizer) = 16P/N bytes

    Trade-off: More communication (all-gather before forward/backward)
    """

    def __init__(self, model: nn.Module, world_size: int):
        self.world_size = world_size
        self.model = model

        # Shard parameters
        self.sharded_params = self._shard_parameters()

        self.dist = SimulatedDistributed(world_size)

    def _shard_parameters(self) -> Dict[int, Dict[str, torch.Tensor]]:
        """
        Shard parameters across GPUs.
        Each GPU stores 1/N of each parameter.
        """
        sharded = {i: {} for i in range(self.world_size)}

        for name, param in self.model.named_parameters():
            flat = param.data.flatten()
            chunk_size = (flat.numel() + self.world_size - 1) // self.world_size

            for i in range(self.world_size):
                start = i * chunk_size
                end = min(start + chunk_size, flat.numel())
                if start < flat.numel():
                    sharded[i][name] = flat[start:end].clone()
                else:
                    sharded[i][name] = torch.zeros(0)

        return sharded

    def all_gather_params(self, rank: int) -> Dict[str, torch.Tensor]:
        """
        All-gather parameters before forward/backward.

        This reconstructs full parameters from shards.
        """
        full_params = {}

        for name, param in self.model.named_parameters():
            # Gather shards from all GPUs
            shards = [self.sharded_params[i][name] for i in range(self.world_size)]

            # Concatenate (simulated all-gather)
            full_flat = torch.cat([s for s in shards if s.numel() > 0])
            full_params[name] = full_flat[:param.numel()].view_as(param)

        return full_params

    def reduce_scatter_grads(self, grads: Dict[str, torch.Tensor], rank: int) -> Dict[str, torch.Tensor]:
        """
        Reduce-scatter gradients after backward.

        Each GPU ends up with 1/N of the reduced gradient.
        """
        local_grads = {}

        for name, grad in grads.items():
            flat = grad.flatten()
            chunk_size = (flat.numel() + self.world_size - 1) // self.world_size

            # Simulated reduce-scatter: sum and take rank's portion
            start = rank * chunk_size
            end = min(start + chunk_size, flat.numel())

            if start < flat.numel():
                local_grads[name] = flat[start:end] / self.world_size

        return local_grads

    def get_memory_per_gpu(self) -> Dict[str, float]:
        """
        Calculate memory per GPU.
        """
        total_params = sum(p.numel() for p in self.model.parameters())

        # FSDP shards everything
        params_per_gpu = total_params / self.world_size

        return {
            'total_params': total_params,
            'params_per_gpu': params_per_gpu,
            'params_bytes_per_gpu': params_per_gpu * 4,  # FP32
            'grads_bytes_per_gpu': params_per_gpu * 4,
            'optimizer_bytes_per_gpu': params_per_gpu * 8,  # Adam
            'total_bytes_per_gpu': params_per_gpu * 16
        }

In [64]:
# Compare DDP vs FSDP memory

print("DDP vs FSDP Memory Comparison")
print("=" * 60)

class BigModel(nn.Module):
    def __init__(self, hidden=2048, layers=24):
        super().__init__()
        self.embed = nn.Linear(1024, hidden)
        self.layers = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(layers)])
        self.head = nn.Linear(hidden, 1000)

    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = F.relu(layer(x))
        return self.head(x)

model = BigModel()
total_params = sum(p.numel() for p in model.parameters())

print(f"Model parameters: {total_params:,} ({total_params * 4 / 1e9:.2f} GB)")

print(f"\n{'GPUs':<8} {'DDP Memory/GPU':>18} {'FSDP Memory/GPU':>18} {'FSDP Reduction':>16}")
print("-" * 65)

for num_gpus in [1, 2, 4, 8]:
    # DDP: Full model on each GPU
    ddp_mem = total_params * 16 / 1e9  # 16 bytes per param (param + grad + optim)

    # FSDP: Sharded
    fsdp = SimulatedFSDP(model, num_gpus)
    fsdp_stats = fsdp.get_memory_per_gpu()
    fsdp_mem = fsdp_stats['total_bytes_per_gpu'] / 1e9

    reduction = ddp_mem / fsdp_mem if fsdp_mem > 0 else 1

    print(f"{num_gpus:<8} {ddp_mem:>17.2f}GB {fsdp_mem:>17.2f}GB {reduction:>15.1f}×")

DDP vs FSDP Memory Comparison
Model parameters: 104,860,648 (0.42 GB)

GPUs         DDP Memory/GPU    FSDP Memory/GPU   FSDP Reduction
-----------------------------------------------------------------
1                     1.68GB              1.68GB             1.0×
2                     1.68GB              0.84GB             2.0×
4                     1.68GB              0.42GB             4.0×
8                     1.68GB              0.21GB             8.0×


In [65]:
# FSDP training pattern

def fsdp_training_pattern():
    pattern = '''
# FSDP Training Pattern
# =====================

from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision

def train_fsdp(rank, world_size):
    setup(rank, world_size)

    # Create model
    model = MyLargeModel()

    # Configure FSDP
    fsdp_config = {
        "sharding_strategy": ShardingStrategy.FULL_SHARD,  # Shard everything
        "mixed_precision": MixedPrecision(
            param_dtype=torch.float16,
            reduce_dtype=torch.float16,
            buffer_dtype=torch.float16,
        ),
        "device_id": rank,
    }

    # Wrap model with FSDP
    model = FSDP(model, **fsdp_config)

    optimizer = torch.optim.AdamW(model.parameters())

    for epoch in range(num_epochs):
        for batch in dataloader:
            optimizer.zero_grad()

            # Forward (FSDP all-gathers params automatically)
            output = model(batch)
            loss = criterion(output, target)

            # Backward (FSDP reduce-scatters grads automatically)
            loss.backward()

            # Update (only local shard is updated)
            optimizer.step()

    cleanup()

# Sharding strategies:
# - FULL_SHARD: Shard params, grads, optimizer (most memory efficient)
# - SHARD_GRAD_OP: Shard grads and optimizer only
# - NO_SHARD: Like DDP (for comparison)
'''
    return pattern

print(fsdp_training_pattern())


# FSDP Training Pattern
# =====================

from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision

def train_fsdp(rank, world_size):
    setup(rank, world_size)
    
    # Create model
    model = MyLargeModel()
    
    # Configure FSDP
    fsdp_config = {
        "sharding_strategy": ShardingStrategy.FULL_SHARD,  # Shard everything
        "mixed_precision": MixedPrecision(
            param_dtype=torch.float16,
            reduce_dtype=torch.float16,
            buffer_dtype=torch.float16,
        ),
        "device_id": rank,
    }
    
    # Wrap model with FSDP
    model = FSDP(model, **fsdp_config)
    
    optimizer = torch.optim.AdamW(model.parameters())
    
    for epoch in range(num_epochs):
        for batch in dataloader:
            optimizer.zero_grad()
            
            # Forward (FSDP all-gathers params automatically)
            output = model(batch)
            loss = crit

Pipeline Parallelism

Split model into stages and process micro-batches in a pipeline.

In [66]:
class PipelineStage(nn.Module):
    """Single stage in a pipeline."""

    def __init__(self, layers: nn.ModuleList, flatten_output: bool = False):
        super().__init__()
        self.layers = layers
        self.flatten_output = flatten_output

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        if self.flatten_output:
            x = torch.flatten(x, 1)
        return x


class GPipeSchedule:
    """
    GPipe-style pipeline schedule.

    Split batch into micro-batches, pipeline through stages.

    Timeline (4 stages, 4 micro-batches):

    Time:     0   1   2   3   4   5   6   7
    Stage 0: F0  F1  F2  F3  B3  B2  B1  B0
    Stage 1:     F0  F1  F2  F3  B3  B2  B1
    Stage 2:         F0  F1  F2  F3  B3  B2
    Stage 3:             F0  F1  F2  F3  B3

    Bubble ratio = (n_stages - 1) / (n_microbatches + n_stages - 1)
    """

    def __init__(self, num_stages: int, num_microbatches: int):
        self.num_stages = num_stages
        self.num_microbatches = num_microbatches

    def get_schedule(self) -> List[Tuple[str, int, int]]:
        """
        Get execution schedule.

        Returns list of (operation, stage, microbatch)
        """
        schedule = []

        # Forward passes
        for mb in range(self.num_microbatches):
            for stage in range(self.num_stages):
                time_slot = mb + stage
                schedule.append(('F', stage, mb, time_slot))

        # Backward passes (reverse order)
        for mb in range(self.num_microbatches - 1, -1, -1):
            for stage in range(self.num_stages - 1, -1, -1):
                time_slot = (self.num_microbatches - 1 - mb) + (self.num_stages - 1 - stage)
                time_slot += self.num_microbatches + self.num_stages - 1
                schedule.append(('B', stage, mb, time_slot))

        # Sort by time slot
        schedule.sort(key=lambda x: (x[3], x[1]))

        return [(op, stage, mb) for op, stage, mb, _ in schedule]

    def get_bubble_ratio(self) -> float:
        """
        Calculate bubble (idle time) ratio.
        """
        total_time = self.num_microbatches + self.num_stages - 1
        bubble_time = self.num_stages - 1
        return bubble_time / total_time


# Visualize pipeline schedule
print("Pipeline Parallelism Schedule (GPipe)")
print("=" * 60)

num_stages = 4
num_microbatches = 8

schedule = GPipeSchedule(num_stages, num_microbatches)
ops = schedule.get_schedule()

print(f"\nStages: {num_stages}, Micro-batches: {num_microbatches}")
print(f"Bubble ratio: {schedule.get_bubble_ratio():.1%}")

# Create timeline visualization
timeline = {s: [] for s in range(num_stages)}
for op, stage, mb in ops:
    timeline[stage].append(f"{op}{mb}")

print(f"\nTimeline (F=Forward, B=Backward, number=microbatch):")
max_len = max(len(t) for t in timeline.values())
for stage in range(num_stages):
    ops_str = ' '.join(f"{op:>3}" for op in timeline[stage])
    print(f"Stage {stage}: {ops_str}")

Pipeline Parallelism Schedule (GPipe)

Stages: 4, Micro-batches: 8
Bubble ratio: 27.3%

Timeline (F=Forward, B=Backward, number=microbatch):
Stage 0:  F0  F1  F2  F3  F4  F5  F6  F7  B7  B6  B5  B4  B3  B2  B1  B0
Stage 1:  F0  F1  F2  F3  F4  F5  F6  F7  B7  B6  B5  B4  B3  B2  B1  B0
Stage 2:  F0  F1  F2  F3  F4  F5  F6  F7  B7  B6  B5  B4  B3  B2  B1  B0
Stage 3:  F0  F1  F2  F3  F4  F5  F6  F7  B7  B6  B5  B4  B3  B2  B1  B0


In [67]:
# Analyze pipeline efficiency

print("\nPipeline Efficiency Analysis")
print("=" * 60)

print(f"\n{'Stages':>8} {'Micro-batches':>15} {'Bubble Ratio':>15} {'Efficiency':>12}")
print("-" * 55)

for stages in [2, 4, 8]:
    for mbs in [4, 8, 16, 32]:
        sched = GPipeSchedule(stages, mbs)
        bubble = sched.get_bubble_ratio()
        efficiency = 1 - bubble
        print(f"{stages:>8} {mbs:>15} {bubble:>14.1%} {efficiency:>11.1%}")


Pipeline Efficiency Analysis

  Stages   Micro-batches    Bubble Ratio   Efficiency
-------------------------------------------------------
       2               4          20.0%       80.0%
       2               8          11.1%       88.9%
       2              16           5.9%       94.1%
       2              32           3.0%       97.0%
       4               4          42.9%       57.1%
       4               8          27.3%       72.7%
       4              16          15.8%       84.2%
       4              32           8.6%       91.4%
       8               4          63.6%       36.4%
       8               8          46.7%       53.3%
       8              16          30.4%       69.6%
       8              32          17.9%       82.1%


# Task
Integrate a real-world image classification dataset (CIFAR-10) into the distributed training examples by:
1.  Loading CIFAR-10 using `torchvision.datasets` and preparing `DataLoader` instances for training and validation.
2.  Adapting the convolutional neural network models to process 3-channel CIFAR-10 images and output predictions for 10 classes.
3.  Updating the `SimulatedDataParallel` class and its demonstration to handle batches from the real dataset.
4.  Refining the `ddp_training_pattern()` and `fsdp_training_pattern()` pseudocode to explicitly demonstrate the usage of `DistributedSampler` with the real dataset and the movement of models and data to GPU devices for distributed training.
5.  Summarize the practical considerations and benefits of these changes.

## Introduce a Real-World Dataset

### Subtask:
Load a standard image classification dataset, such as CIFAR-10, using `torchvision.datasets`. This will involve downloading the dataset and setting up `DataLoader` instances for training and validation.


**Reasoning**:
The first step is to define the necessary transformations for the CIFAR-10 dataset, as specified in the instructions. This includes converting images to PyTorch tensors and normalizing them with the given mean and standard deviation.



In [68]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10 training dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# Load CIFAR-10 test (validation) dataset
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Create DataLoader instances
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"CIFAR-10 training dataset size: {len(train_dataset)}")
print(f"CIFAR-10 test dataset size: {len(test_dataset)}")
print(f"Batch size for DataLoaders: {batch_size}")

CIFAR-10 training dataset size: 50000
CIFAR-10 test dataset size: 10000
Batch size for DataLoaders: 64


**Reasoning**:
The current `SimpleModel` is a fully connected network, which is not ideal for image data like CIFAR-10. I will create a new `CifarCNN` model with convolutional layers that are designed to handle 3-channel 32x32 images and output 10 classes, aligning with the CIFAR-10 dataset specifications.



In [69]:
class CifarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: 3x32x32
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        # Output: 32x32x32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        # Output after pooling: 64x16x16
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        # Output after pooling: 128x8x8
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(128 * 8 * 8, 10)  # 10 classes for CIFAR-10

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1) # Flatten all dimensions except batch
        x = self.fc(x)
        return x

print("Defined CifarCNN model suitable for CIFAR-10 data.")

Defined CifarCNN model suitable for CIFAR-10 data.


**Reasoning**:
The previous `SimulatedDataParallel` demonstration used a `SimpleModel` and randomly generated data. To adapt it for CIFAR-10, I need to instantiate the newly defined `CifarCNN` model, fetch actual batches from the `train_loader`, and ensure the forward, loss calculation, and backward/sync operations correctly handle the image data and 10-class output.



In [70]:
print("Pipeline Parallelism with CifarCNN")
print("=" * 60)

# Define the layers of CifarCNN so they can be grouped into PipelineStages
class CifarCNN_Pipeline(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.fc = nn.Linear(128 * 4 * 4, 10)

    def forward(self, x):
        # This forward is just for getting sequential layers, not for direct use
        pass

# Instantiate the individual layers of CifarCNN_Pipeline
full_model_layers = CifarCNN_Pipeline()

# Group layers into stages
# For demonstration, let's assume 2 GPUs/stages
# Stage 0: conv1, bn1, pool1, conv2, bn2, pool2
# Stage 1: conv3, bn3, pool3, flatten, fc

# Helper function to get model output shape after convolutions and pooling
def get_conv_output_shape(model, input_shape):
    dummy_input = torch.rand(1, *input_shape)
    x = model.conv1(dummy_input)
    x = model.pool(x)
    x = model.conv2(x)
    x = model.pool(x)
    x = model.conv3(x)
    x = model.pool(x)
    return x.shape[1:] # return C, H, W

input_shape = (3, 32, 32) # CIFAR-10 image size

stage0_layers = nn.ModuleList([
    full_model_layers.conv1,
    full_model_layers.bn1,
    nn.ReLU(),
    full_model_layers.pool,
    full_model_layers.conv2,
    full_model_layers.bn2,
    nn.ReLU(),
    full_model_layers.pool,
])

stage1_layers = nn.ModuleList([
    full_model_layers.conv3,
    full_model_layers.bn3,
    nn.ReLU(),
    full_model_layers.pool,
    nn.Flatten(), # Added Flatten layer here
    full_model_layers.fc
])

# Determine if the last layer needs flattening
# We will do this manually for the second stage to show the flatten_output flag

# Simulate a multi-GPU setup
world_size = 2

# Create PipelineStage instances
stage0 = PipelineStage(stage0_layers)
stage1 = PipelineStage(stage1_layers, flatten_output=False) # flatten_output is now handled by nn.Flatten

# Get a batch from the CIFAR-10 training loader
dataiter = iter(train_loader)
images, labels = next(dataiter)

print(f"\nInput image batch shape: {images.shape}")

# Simulate forward pass through pipeline stages
# In a real scenario, these would be on different GPUs and communicate

# Stage 0 forward
output_stage0 = stage0(images)
print(f"Output shape after Stage 0: {output_stage0.shape}")

# Simulate transfer to next GPU
# output_stage0 = output_stage0.to(f'cuda:{rank+1}')

# Stage 1 forward
output_stage1 = stage1(output_stage0)
print(f"Output shape after Stage 1 (flattened): {output_stage1.shape}")

print("\nSimulated pipeline execution for CifarCNN complete.")

Pipeline Parallelism with CifarCNN

Input image batch shape: torch.Size([64, 3, 32, 32])
Output shape after Stage 0: torch.Size([64, 64, 8, 8])
Output shape after Stage 1 (flattened): torch.Size([64, 10])

Simulated pipeline execution for CifarCNN complete.


**Reasoning**:
The `RuntimeError` indicates a mismatch in the input feature dimensions for the final linear layer (`self.fc`) in the `CifarCNN` model. After three `MaxPool2d` operations, the spatial dimensions become 4x4, not 8x8. Therefore, the `in_features` for the `nn.Linear` layer should be `128 * 4 * 4` instead of `128 * 8 * 8`.



In [71]:
class CifarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: 3x32x32
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        # Output after 1st pool: 32x16x16
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        # Output after 2nd pool: 64x8x8
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        # Output after 3rd pool: 128x4x4
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(128 * 4 * 4, 10)  # Corrected input features for CIFAR-10

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1) # Flatten all dimensions except batch
        x = self.fc(x)
        return x

print("Defined CifarCNN model suitable for CIFAR-10 data.")

Defined CifarCNN model suitable for CIFAR-10 data.


**Reasoning**:
The `CifarCNN` model was redefined in the previous step to fix the input feature dimension error. Now, I need to re-instantiate the corrected `CifarCNN` model and rerun the `SimulatedDataParallel` demonstration to verify that the data parallelism logic works correctly with the CIFAR-10 data and the updated model.



In [72]:
print("Simulated Data Parallelism with CIFAR-10 (After CifarCNN Correction)")
print("=" * 75)

# Instantiate the CORRECTED CifarCNN model
model = CifarCNN()
world_size = 4
batch_size = 64  # Total batch size

dp = SimulatedDataParallel(model, world_size)

# Get a batch from the CIFAR-10 training loader
dataiter = iter(train_loader)
images, labels = next(dataiter)

print(f"Input images shape: {images.shape} (Total batch size {batch_size})")

# Forward pass (simulated splitting of batch across GPUs)
outputs = dp.forward(images)

# Calculate expected chunk size for outputs
expected_chunk_size = batch_size // world_size
print(f"Outputs per GPU (expected chunk size {expected_chunk_size}): {[o.shape[0] for o in outputs]}")

# Compute losses for each replica
criterion = nn.CrossEntropyLoss()
losses = []
for i, output in enumerate(outputs):
    start = i * expected_chunk_size
    end = start + expected_chunk_size
    if i == world_size - 1: # Handle potential remainder for last chunk if batch_size is not perfectly divisible
        target_labels = labels[start:]
    else:
        target_labels = labels[start:end]

    loss = criterion(output, target_labels)
    losses.append(loss)
    print(f"GPU {i} loss: {loss.item():.4f}")

# Backward and sync gradients
dp.backward_and_sync(losses)

# Verify gradients are synchronized
print("\nGradients synchronized across GPUs:")
for name, _ in dp.models[0].named_parameters():
    grads = [m.get_parameter(name).grad for m in dp.models]
    if all(g is not None for g in grads):
        all_equal = all(torch.allclose(grads[0], g) for g in grads[1:])
        print(f"  {name}: {'✓' if all_equal else '✗'}")
    else:
        print(f"  {name}: Gradients not found for all replicas.")

Simulated Data Parallelism with CIFAR-10 (After CifarCNN Correction)
Input images shape: torch.Size([64, 3, 32, 32]) (Total batch size 64)
Outputs per GPU (expected chunk size 16): [16, 16, 16, 16]
GPU 0 loss: 2.3166
GPU 1 loss: 2.3199
GPU 2 loss: 2.6100
GPU 3 loss: 2.5647

Gradients synchronized across GPUs:
  conv1.weight: ✓
  conv1.bias: ✓
  bn1.weight: ✓
  bn1.bias: ✓
  conv2.weight: ✓
  conv2.bias: ✓
  bn2.weight: ✓
  bn2.bias: ✓
  conv3.weight: ✓
  conv3.bias: ✓
  bn3.weight: ✓
  bn3.bias: ✓
  fc.weight: ✓
  fc.bias: ✓


## Enhance DDP/FSDP Training Patterns with Real Data

### Subtask:
Refine the `ddp_training_pattern()` and `fsdp_training_pattern()` pseudocode examples to explicitly show how `DistributedSampler` would be used with the CIFAR-10 dataset and how models and data are moved to GPU devices for distributed training.


**Reasoning**:
The subtask requires modifying the `ddp_training_pattern()` pseudocode to incorporate CIFAR-10 related elements, including the `CifarCNN` model, `DistributedSampler`, explicit device placement, and dummy variables. This step involves updating the existing `ddp_training_pattern` function in the notebook.



In [73]:
def ddp_training_pattern():
    """
    Show typical DDP training pattern (pseudo-code for multi-GPU) with CIFAR-10.
    """
    pattern = '''
# DDP Training Pattern with CIFAR-10
# =================================

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms
import os

# Dummy CifarCNN model for demonstration (should be defined externally in a real scenario)
class CifarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(128 * 4 * 4, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

def setup(rank, world_size):
    """Initialize distributed process group."""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

def cleanup():
    dist.destroy_process_group()

def train(rank, world_size):
    setup(rank, world_size)

    # Dummy dataset, batch_size, criterion, num_epochs for pseudocode clarity
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    dataset = datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
    batch_size = 64 # Per-GPU batch size
    criterion = nn.CrossEntropyLoss()
    num_epochs = 2

    # Create model and move to GPU
    model = CifarCNN().to(rank)

    # Wrap with DDP
    ddp_model = DDP(model, device_ids=[rank])

    # Use DistributedSampler for data
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    dataloader = DataLoader(dataset, sampler=sampler, batch_size=batch_size, num_workers=2)

    optimizer = torch.optim.Adam(ddp_model.parameters())

    for epoch in range(num_epochs):
        sampler.set_epoch(epoch)  # Important for shuffling

        for batch_idx, (data, target) in enumerate(dataloader):
            optimizer.zero_grad()

            # Move data to GPU
            data, target = data.to(rank), target.to(rank)

            # Forward (each GPU processes 1/N of batch)
            output = ddp_model(data)
            loss = criterion(output, target)

            # Backward (gradients auto-synced by DDP)
            loss.backward()

            # Update (all GPUs have same gradients)
            optimizer.step()

            if batch_idx % 100 == 0:
                print(f"Rank {rank}, Epoch {epoch}, Batch {batch_idx}: Loss {loss.item():.4f}")

    cleanup()

# Launch with torch.multiprocessing
if __name__ == "__main__":
    # This part would typically be in a separate script or managed by a launcher
    # For pseudocode, we show the intention.
    # world_size = torch.cuda.device_count()
    # if world_size > 0:
    #     torch.multiprocessing.spawn(train, args=(world_size,), nprocs=world_size)
    # else:
    #     print("No CUDA devices found, DDP simulation cannot be run.")
    print("DDP pseudocode defined. To run, use torch.multiprocessing.spawn in a real environment.")
'''
    return pattern

# Redefine the function with updated pseudocode
ddp_training_pattern = ddp_training_pattern()
print(ddp_training_pattern)


# DDP Training Pattern with CIFAR-10
# =================================

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms
import os

# Dummy CifarCNN model for demonstration (should be defined externally in a real scenario)
class CifarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(128 * 4 * 4, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.con

**Reasoning**:
The subtask requires modifying the `fsdp_training_pattern()` pseudocode to incorporate CIFAR-10 related elements, including the `CifarCNN` model, `DistributedSampler`, explicit device placement, and dummy variables. This step involves updating the existing `fsdp_training_pattern` function in the notebook.



In [74]:
def fsdp_training_pattern():
    pattern = '''
# FSDP Training Pattern with CIFAR-10
# =====================

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms
import os

# Dummy CifarCNN model for demonstration (should be defined externally in a real scenario)
class CifarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(128 * 4 * 4, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

def setup(rank, world_size):
    """Initialize distributed process group."""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

def cleanup():
    dist.destroy_process_group()

def train_fsdp(rank, world_size):
    setup(rank, world_size)

    # Dummy dataset, batch_size, criterion, num_epochs for pseudocode clarity
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    dataset = datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
    batch_size = 64 # Per-GPU batch size
    criterion = nn.CrossEntropyLoss()
    num_epochs = 2

    # Create model and move to GPU
    model = CifarCNN().to(rank)

    # Configure FSDP
    fsdp_config = {
        "sharding_strategy": ShardingStrategy.FULL_SHARD,  # Shard everything
        "mixed_precision": MixedPrecision(
            param_dtype=torch.float16,
            reduce_dtype=torch.float16,
            buffer_dtype=torch.float16,
        ),
        "device_id": rank,
    }

    # Wrap model with FSDP
    model = FSDP(model, **fsdp_config)

    # Use DistributedSampler for data
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    dataloader = DataLoader(dataset, sampler=sampler, batch_size=batch_size, num_workers=2)

    optimizer = torch.optim.AdamW(model.parameters())

    for epoch in range(num_epochs):
        sampler.set_epoch(epoch)  # Important for shuffling

        for batch_idx, (data, target) in enumerate(dataloader):
            optimizer.zero_grad()

            # Move data to GPU
            data, target = data.to(rank), target.to(rank)

            # Forward (FSDP all-gathers params automatically)
            output = model(data)
            loss = criterion(output, target)

            # Backward (FSDP reduce-scatters grads automatically)
            loss.backward()

            # Update (only local shard is updated)
            optimizer.step()

            if batch_idx % 100 == 0:
                print(f"Rank {rank}, Epoch {epoch}, Batch {batch_idx}: Loss {loss.item():.4f}")

    cleanup()

# Sharding strategies:
# - FULL_SHARD: Shard params, grads, optimizer (most memory efficient)
# - SHARD_GRAD_OP: Shard grads and optimizer only
# - NO_SHARD: Like DDP (for comparison)

# Launch with torch.multiprocessing
if __name__ == "__main__":
    # This part would typically be in a separate script or managed by a launcher
    # For pseudocode, we show the intention.
    # world_size = torch.cuda.device_count()
    # if world_size > 0:
    #     torch.multiprocessing.spawn(train_fsdp, args=(world_size,), nprocs=world_size)
    # else:
    #     print("No CUDA devices found, FSDP simulation cannot be run.")
    print("FSDP pseudocode defined. To run, use torch.multiprocessing.spawn in a real environment.")
'''
    return pattern

# Redefine the function with updated pseudocode
fsdp_training_pattern = fsdp_training_pattern()
print(fsdp_training_pattern)



# FSDP Training Pattern with CIFAR-10
# =====================

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms
import os

# Dummy CifarCNN model for demonstration (should be defined externally in a real scenario)
class CifarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(128 * 4 * 4, 10)

    def forw